# Credit Default Exploratory Analysis

## Project Overview
Analyze borrower and repayment data to identify patterns associated with default risk. This is a **data analysis** project — we focus on distributions, correlations, and interpretable risk drivers rather than building prediction models.

## Learning Objectives
- Explore borrower demographics and financial features in a credit dataset
- Identify distributions and correlations associated with default
- Segment borrowers by risk profile using descriptive statistics
- Derive interpretable risk indicators for business stakeholders

## Problem Statement
A lending institution wants to understand which borrower characteristics are most associated with loan default so they can improve underwriting guidelines and risk screening criteria.

## Why This Project Matters
Credit default is one of the largest sources of financial loss for banks and lenders. Even small improvements in understanding default patterns can save millions in losses and enable fairer, more targeted lending decisions.

## Dataset Overview
Synthetic credit dataset: ~5K borrowers with demographic features (age, income, employment), loan features (amount, term, interest rate, purpose), and default status. Patterns are modeled on typical consumer lending data.

## Dataset Source and License Notes
- Synthetic data generated for educational purposes
- Patterns inspired by public lending datasets (e.g., Lending Club, Home Credit)
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 5000

age = np.clip(np.random.normal(38, 12, n).astype(int), 21, 70)
income = np.clip(np.random.lognormal(10.5, 0.6, n).astype(int), 15000, 300000)
employment_years = np.clip(np.random.exponential(5, n).astype(int), 0, 35)
home_ownership = np.random.choice(['Own', 'Mortgage', 'Rent'], n, p=[0.25, 0.40, 0.35])
loan_amount = np.clip(np.random.lognormal(9.5, 0.7, n).astype(int), 1000, 100000)
loan_term = np.random.choice([12, 24, 36, 48, 60], n, p=[0.10, 0.15, 0.35, 0.25, 0.15])
interest_rate = np.clip(np.random.normal(12, 4, n), 4, 28).round(1)
purpose = np.random.choice(['Debt Consolidation', 'Home Improvement', 'Medical', 'Education',
                             'Business', 'Auto', 'Other'], n, p=[0.30, 0.15, 0.10, 0.10, 0.12, 0.13, 0.10])
credit_score = np.clip(np.random.normal(680, 60, n).astype(int), 400, 850)
dti_ratio = np.clip(np.random.normal(20, 8, n), 2, 50).round(1)
num_credit_lines = np.clip(np.random.poisson(5, n), 0, 20)

# Default probability driven by risk factors
log_odds = (-3.0
    + 0.03 * (25 - np.minimum(employment_years, 25))
    + 0.8 * (dti_ratio > 30).astype(float)
    + 1.2 * (credit_score < 620).astype(float)
    + 0.5 * (interest_rate > 18).astype(float)
    + 0.3 * (loan_amount > 50000).astype(float)
    - 0.3 * (income > 80000).astype(float)
    + np.random.normal(0, 0.5, n)
)
default_prob = 1 / (1 + np.exp(-log_odds))
defaulted = (np.random.random(n) < default_prob).astype(int)

df = pd.DataFrame({
    'Age': age, 'Income': income, 'EmploymentYears': employment_years,
    'HomeOwnership': home_ownership, 'LoanAmount': loan_amount,
    'LoanTerm': loan_term, 'InterestRate': interest_rate,
    'Purpose': purpose, 'CreditScore': credit_score,
    'DTI_Ratio': dti_ratio, 'NumCreditLines': num_credit_lines,
    'Defaulted': defaulted,
})
print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["Defaulted"].mean():.1%}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Credit score range: {df["CreditScore"].min()} - {df["CreditScore"].max()}')
print(f'Income range: ${df["Income"].min():,} - ${df["Income"].max():,}')
print(f'Loan amount range: ${df["LoanAmount"].min():,} - ${df["LoanAmount"].max():,}')
print(f'\nDefault distribution:\n{df["Defaulted"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

axes[0,0].hist(df['CreditScore'], bins=40, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Credit Score Distribution')

axes[0,1].hist(df['Income'], bins=40, edgecolor='black', alpha=0.7, color='coral')
axes[0,1].set_title('Income Distribution')
axes[0,1].set_xlabel('Income ($)')

axes[0,2].hist(df['LoanAmount'], bins=40, edgecolor='black', alpha=0.7, color='green')
axes[0,2].set_title('Loan Amount Distribution')

axes[1,0].hist(df['InterestRate'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1,0].set_title('Interest Rate Distribution')

axes[1,1].hist(df['DTI_Ratio'], bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[1,1].set_title('DTI Ratio Distribution')

df['Purpose'].value_counts().plot.barh(ax=axes[1,2], edgecolor='black', color='steelblue')
axes[1,2].set_title('Loan Purpose')
axes[1,2].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Target Analysis — Default Rate by Key Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Default rate by credit score bucket
df['CreditBucket'] = pd.cut(df['CreditScore'], bins=[400, 580, 620, 680, 740, 850],
                              labels=['<580','580-620','620-680','680-740','740+'])
df.groupby('CreditBucket')['Defaulted'].mean().plot.bar(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Default Rate by Credit Score')
axes[0,0].set_ylabel('Default Rate')
axes[0,0].tick_params(axis='x', rotation=0)

# Default rate by DTI bucket
df['DTI_Bucket'] = pd.cut(df['DTI_Ratio'], bins=[0, 10, 20, 30, 50],
                            labels=['<10%', '10-20%', '20-30%', '30%+'])
df.groupby('DTI_Bucket')['Defaulted'].mean().plot.bar(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Default Rate by DTI Ratio')
axes[0,1].tick_params(axis='x', rotation=0)

# Default rate by purpose
df.groupby('Purpose')['Defaulted'].mean().sort_values(ascending=True).plot.barh(
    ax=axes[1,0], edgecolor='black', color='mediumpurple')
axes[1,0].set_title('Default Rate by Loan Purpose')

# Default rate by home ownership
df.groupby('HomeOwnership')['Defaulted'].mean().plot.bar(ax=axes[1,1], edgecolor='black', color='green')
axes[1,1].set_title('Default Rate by Home Ownership')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'default_rates.png'), dpi=100, bbox_inches='tight')
plt.show()

## Correlation Analysis

In [ ]:
numeric_cols = ['Age', 'Income', 'EmploymentYears', 'LoanAmount', 'LoanTerm',
                'InterestRate', 'CreditScore', 'DTI_Ratio', 'NumCreditLines', 'Defaulted']
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'correlation.png'), dpi=100, bbox_inches='tight')
plt.show()

print('\nCorrelation with Default:')
print(corr['Defaulted'].drop('Defaulted').sort_values(ascending=False).round(3))

## Risk Segmentation

In [ ]:
def risk_tier(row):
    if row['CreditScore'] < 620 or row['DTI_Ratio'] > 30:
        return 'High Risk'
    elif row['CreditScore'] < 680 or row['DTI_Ratio'] > 20:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['RiskTier'] = df.apply(risk_tier, axis=1)
risk_summary = df.groupby('RiskTier').agg(
    count=('Defaulted', 'count'),
    default_rate=('Defaulted', 'mean'),
    avg_income=('Income', 'mean'),
    avg_credit_score=('CreditScore', 'mean'),
    avg_loan_amount=('LoanAmount', 'mean'),
    avg_interest_rate=('InterestRate', 'mean'),
).round(3)
risk_summary['pct_borrowers'] = (risk_summary['count'] / risk_summary['count'].sum() * 100).round(1)
print('Risk Tier Summary:')
print(risk_summary)

## Default vs Non-Default Profile Comparison

In [ ]:
profile = df.groupby('Defaulted')[numeric_cols[:-1]].mean()
profile.index = ['No Default', 'Default']
print('Average Profile — Default vs No Default:')
print(profile.T.round(2))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['CreditScore', 'DTI_Ratio', 'InterestRate']):
    for label, grp in df.groupby('Defaulted'):
        ax.hist(grp[col], bins=30, alpha=0.5, label='Default' if label else 'No Default', edgecolor='black')
    ax.set_title(f'{col} by Default Status')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'default_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interest Rate vs Default Deep Dive

In [ ]:
df['RateBucket'] = pd.cut(df['InterestRate'], bins=[0, 8, 12, 16, 20, 30],
                           labels=['<8%', '8-12%', '12-16%', '16-20%', '20%+'])
rate_default = df.groupby('RateBucket').agg(
    count=('Defaulted', 'count'),
    default_rate=('Defaulted', 'mean'),
    avg_credit_score=('CreditScore', 'mean'),
).round(3)
print('Default Rate by Interest Rate:')
print(rate_default)

fig, ax = plt.subplots(figsize=(8, 5))
rate_default['default_rate'].plot.bar(ax=ax, edgecolor='black', color='coral')
ax.set_title('Default Rate by Interest Rate Bucket')
ax.set_ylabel('Default Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'rate_default.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Credit Score < 620** is the strongest single default predictor — these borrowers default at 2-3x the rate of prime borrowers
- **DTI > 30%** significantly increases default risk, indicating borrowers are overextended
- **Interest Rate > 18%** correlates with higher default — partly because high rates are assigned to risky borrowers, creating a feedback loop
- **Employment tenure** matters: shorter employment correlates with higher default
- **Loan purpose** shows modest variation, with some purposes (e.g., debt consolidation at high DTI) showing elevated risk
- The risk tier segmentation separates the portfolio into actionable groups for differential treatment

## Limitations
- Synthetic data with engineered default drivers — real credit data is messier and more nuanced
- No temporal dimension (vintage analysis, payment history over time)
- No external features (macro economy, regional factors, bureau data beyond credit score)
- Binary default doesn't capture severity (partial default, cure rates, loss given default)

## How to Improve This Project
- Use real anonymized credit data for more realistic patterns
- Add vintage/cohort analysis to track default rates over loan lifetime
- Incorporate payment history and delinquency progression
- Build logistic regression or gradient boosting models to quantify risk factor importance
- Add fairness analysis across demographic groups

## Production Considerations
- Risk tier definitions should be calibrated regularly against actual default rates
- Monitor portfolio concentration risk (too many loans in one risk tier)
- Track default rate trends by vintage to catch underwriting degradation early
- Regulatory compliance requires transparent, explainable risk criteria

## Common Mistakes
- Using correlation alone to infer causation (high interest rates don't *cause* default)
- Ignoring the interaction between features (low credit score + high DTI is much worse than either alone)
- Not accounting for survivorship bias (only analyzing originated loans, not declined applicants)
- Setting binary cutoffs without considering the cost trade-off of false positives vs false negatives

## Mini Challenge / Exercises
1. Create a 3-way segmentation: Credit Score × DTI × Employment Years. Which combination has the highest default rate?
2. Calculate the expected loss for each risk tier assuming 40% loss-given-default.
3. If the lender tightened underwriting to reject all High Risk borrowers, how much revenue would be lost vs losses avoided?
4. Find the credit score threshold that balances default rate < 10% with maximum loan volume.

## Final Summary / Key Takeaways
- Credit default analysis is fundamentally about identifying who can't afford to repay
- Credit score, DTI ratio, and interest rate are the top three risk indicators
- Risk segmentation enables differentiated policies: stricter terms for high-risk, better rates for low-risk
- The relationship between features matters more than individual features — context is key
- Any credit analysis must consider fairness, transparency, and regulatory requirements